---

## **`Batch HU Analysis for CMC Dataset`**

================================================

### Analyzes HU distributions across all patients in the dataset directory and provides aggregate statistics for window parameter validation.

Author: Hasan Shaikh

In [6]:
# Set your dataset directory path here
dataset_dir = "/home/radiomicsserver/Downloads/hn_cnn-main/convert_nifti_format/"

# Run the analysis
df, stats = batch_analyze_dataset(dataset_dir, output_dir="./hu_analysis_results")

CMC DATASET BATCH HU ANALYSIS

[1/5] Scanning directory: /home/radiomicsserver/Downloads/hn_cnn-main/convert_nifti_format/
✓ Found 102 patients with CT and mask files

[2/5] Analyzing HU distributions for all patients...


Processing patients: 100%|██████████| 102/102 [02:06<00:00,  1.24s/it]


✓ Successfully analyzed 102 patients

✓ Saved detailed results: ./hu_analysis_results/patient_hu_statistics.csv

[3/5] Calculating aggregate statistics...

AGGREGATE STATISTICS (ALL PATIENTS)

Cohort size: 102 patients
Average tumor volume: 4833 ± 5017 voxels

Tumor HU values (cohort average):
  Mean:              91.35 ± 30.09 HU
  Median:            99.00 HU
  Range:             [-1024.0, 2309.0] HU
  IQR (25th-75th):   [72.7, 115.6] HU

Within-patient HU variability:
  Average std:       66.05 HU

--------------------------------------------------------------------------------
INTERPRETATION:
--------------------------------------------------------------------------------
⚠ Tumor HU mean (91.4) is outside typical range
⚠ Wide HU range may include non-tumor tissue

[4/5] Testing window parameters across cohort...

WINDOW PARAMETER ANALYSIS

Best (-50 to 300 HU):
  Range: (-50, 300)
  Mean tumor coverage: 97.3% ± 4.2%
  Minimum coverage: 79.1%
  Patients with ≥90% coverage: 93
  Patie

In [ ]:
"""
Batch HU Analysis for CMC Dataset
==================================
"""

import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from pathlib import Path
import pandas as pd
from tqdm import tqdm

try:
    import nibabel as nib
    NIBABEL_AVAILABLE = True
except ImportError:
    NIBABEL_AVAILABLE = False
    print("ERROR: nibabel is required. Install with: pip install nibabel")


def find_patient_files(root_dir):
    """Find all patient CT and mask files in directory structure"""
    patient_files = []
    
    # Look for patient folders
    patient_folders = sorted([d for d in Path(root_dir).iterdir() if d.is_dir()])
    
    for patient_folder in patient_folders:
        patient_id = patient_folder.name
        
        # Look for image files (prefer resampled version)
        image_path = None
        if (patient_folder / 'image_re.nii.gz').exists():
            image_path = patient_folder / 'image_re.nii.gz'
        elif (patient_folder / 'image.nii.gz').exists():
            image_path = patient_folder / 'image.nii.gz'
        
        # Look for mask files (prefer resampled version)
        mask_path = None
        mask_files = list(patient_folder.glob('mask*_re.nii.gz'))
        if mask_files:
            mask_path = mask_files[0]
        else:
            mask_files = list(patient_folder.glob('mask*.nii.gz'))
            if mask_files:
                mask_path = mask_files[0]
        
        if image_path and mask_path:
            patient_files.append({
                'patient_id': patient_id,
                'image_path': str(image_path),
                'mask_path': str(mask_path)
            })
    
    return patient_files


def analyze_single_patient(image_path, mask_path, patient_id):
    """Analyze HU distribution for a single patient"""
    try:
        # Load image
        img = nib.load(image_path)
        ct_data = img.get_fdata()
        
        # Load mask
        mask = nib.load(mask_path)
        mask_data = mask.get_fdata()
        
        # Get HU values from tumor region
        tumor_pixels = ct_data[mask_data > 0]
        
        if len(tumor_pixels) == 0:
            return None
        
        # Calculate statistics
        results = {
            'patient_id': patient_id,
            'tumor_volume_voxels': len(tumor_pixels),
            'hu_mean': np.mean(tumor_pixels),
            'hu_std': np.std(tumor_pixels),
            'hu_min': np.min(tumor_pixels),
            'hu_max': np.max(tumor_pixels),
            'hu_median': np.median(tumor_pixels),
            'hu_q25': np.percentile(tumor_pixels, 25),
            'hu_q75': np.percentile(tumor_pixels, 75),
            'hu_p01': np.percentile(tumor_pixels, 1),
            'hu_p99': np.percentile(tumor_pixels, 99),
        }
        
        # Test different windows
        windows = {
            'best': (-50, 300),
            'wide': (-500, 500),
            'medium': (-250, 250),
        }
        
        for window_name, (min_hu, max_hu) in windows.items():
            in_window = np.sum((tumor_pixels >= min_hu) & (tumor_pixels <= max_hu))
            pct_in_window = (in_window / len(tumor_pixels)) * 100
            results[f'pct_in_{window_name}_window'] = pct_in_window
        
        # Store HU values for histogram (sample if too large)
        if len(tumor_pixels) > 10000:
            results['hu_sample'] = np.random.choice(tumor_pixels, 10000, replace=False)
        else:
            results['hu_sample'] = tumor_pixels
        
        return results
        
    except Exception as e:
        print(f"  Error processing {patient_id}: {str(e)}")
        return None


def batch_analyze_dataset(root_dir, output_dir='./hu_analysis_results'):
    """
    Analyze entire dataset
    
    Parameters:
    -----------
    root_dir : str
        Root directory containing patient folders (e.g., /path/to/convert_nifti_format/)
    output_dir : str
        Directory to save results
        
    Returns:
    --------
    df : pandas DataFrame
        Results dataframe
    aggregate_stats : dict
        Aggregate statistics
    """
    print("="*80)
    print("CMC DATASET BATCH HU ANALYSIS")
    print("="*80)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Find all patient files
    print(f"\n[1/5] Scanning directory: {root_dir}")
    patient_files = find_patient_files(root_dir)
    print(f"✓ Found {len(patient_files)} patients with CT and mask files")
    
    if len(patient_files) == 0:
        print("\nERROR: No patient files found!")
        print("Make sure the directory contains patient folders with:")
        print("  - image.nii.gz or image_re.nii.gz")
        print("  - mask*.nii.gz or mask*_re.nii.gz")
        return None, None
    
    # Analyze each patient
    print(f"\n[2/5] Analyzing HU distributions for all patients...")
    all_results = []
    all_hu_samples = []
    
    for pf in tqdm(patient_files, desc="Processing patients"):
        result = analyze_single_patient(
            pf['image_path'], 
            pf['mask_path'], 
            pf['patient_id']
        )
        
        if result:
            all_results.append(result)
            all_hu_samples.extend(result['hu_sample'])
    
    print(f"✓ Successfully analyzed {len(all_results)} patients")
    
    # Convert to DataFrame
    df = pd.DataFrame(all_results)
    
    # Remove 'hu_sample' column before saving CSV
    df_to_save = df.drop('hu_sample', axis=1)
    
    # Save detailed results
    csv_path = os.path.join(output_dir, 'patient_hu_statistics.csv')
    df_to_save.to_csv(csv_path, index=False)
    print(f"\n✓ Saved detailed results: {csv_path}")
    
    # Calculate aggregate statistics
    print(f"\n[3/5] Calculating aggregate statistics...")
    aggregate_stats = calculate_aggregate_stats(df)
    print_aggregate_stats(aggregate_stats)
    
    # Test window parameters
    print(f"\n[4/5] Testing window parameters across cohort...")
    window_analysis = analyze_window_performance(df)
    print_window_analysis(window_analysis)
    
    # Generate visualizations
    print(f"\n[5/5] Generating visualizations...")
    generate_visualizations(df, all_hu_samples, output_dir)
    
    # Save summary report
    save_summary_report(aggregate_stats, window_analysis, output_dir)
    
    print("\n" + "="*80)
    print("ANALYSIS COMPLETE")
    print("="*80)
    print(f"\nResults saved to: {output_dir}/")
    print("  - patient_hu_statistics.csv")
    print("  - aggregate_statistics.txt")
    print("  - hu_distribution_histogram.png")
    print("  - window_comparison.png")
    print("  - patient_variability.png")
    
    return df, aggregate_stats


def calculate_aggregate_stats(df):
    """Calculate aggregate statistics across all patients"""
    stats = {
        'n_patients': len(df),
        'mean_tumor_volume': df['tumor_volume_voxels'].mean(),
        'std_tumor_volume': df['tumor_volume_voxels'].std(),
        
        'cohort_hu_mean': df['hu_mean'].mean(),
        'cohort_hu_std': df['hu_mean'].std(),
        'cohort_hu_min': df['hu_min'].min(),
        'cohort_hu_max': df['hu_max'].max(),
        'cohort_hu_median': df['hu_median'].median(),
        
        'avg_within_patient_std': df['hu_std'].mean(),
        
        'cohort_hu_q25': df['hu_q25'].mean(),
        'cohort_hu_q75': df['hu_q75'].mean(),
    }
    
    return stats


def print_aggregate_stats(stats):
    """Print aggregate statistics"""
    print("\n" + "="*80)
    print("AGGREGATE STATISTICS (ALL PATIENTS)")
    print("="*80)
    
    print(f"\nCohort size: {stats['n_patients']} patients")
    print(f"Average tumor volume: {stats['mean_tumor_volume']:.0f} ± {stats['std_tumor_volume']:.0f} voxels")
    
    print(f"\nTumor HU values (cohort average):")
    print(f"  Mean:              {stats['cohort_hu_mean']:.2f} ± {stats['cohort_hu_std']:.2f} HU")
    print(f"  Median:            {stats['cohort_hu_median']:.2f} HU")
    print(f"  Range:             [{stats['cohort_hu_min']:.1f}, {stats['cohort_hu_max']:.1f}] HU")
    print(f"  IQR (25th-75th):   [{stats['cohort_hu_q25']:.1f}, {stats['cohort_hu_q75']:.1f}] HU")
    
    print(f"\nWithin-patient HU variability:")
    print(f"  Average std:       {stats['avg_within_patient_std']:.2f} HU")
    
    print("\n" + "-"*80)
    print("INTERPRETATION:")
    print("-"*80)
    
    if 20 <= stats['cohort_hu_mean'] <= 60:
        print("✓ Tumor HU mean is in expected range for soft tissue (20-60 HU)")
    else:
        print(f"⚠ Tumor HU mean ({stats['cohort_hu_mean']:.1f}) is outside typical range")
    
    if stats['cohort_hu_min'] >= -100 and stats['cohort_hu_max'] <= 200:
        print("✓ HU range suggests predominantly soft tissue tumors")
    else:
        print(f"⚠ Wide HU range may include non-tumor tissue")


def analyze_window_performance(df):
    """Analyze window parameter performance"""
    analysis = {
        'best_window': {
            'name': 'Best (-50 to 300 HU)',
            'range': (-50, 300),
            'mean_coverage': df['pct_in_best_window'].mean(),
            'std_coverage': df['pct_in_best_window'].std(),
            'min_coverage': df['pct_in_best_window'].min(),
            'patients_above_90': (df['pct_in_best_window'] >= 90).sum(),
            'patients_above_95': (df['pct_in_best_window'] >= 95).sum(),
        },
        'wide_window': {
            'name': 'Wide (-500 to 500 HU)',
            'range': (-500, 500),
            'mean_coverage': df['pct_in_wide_window'].mean(),
            'std_coverage': df['pct_in_wide_window'].std(),
            'min_coverage': df['pct_in_wide_window'].min(),
        },
        'medium_window': {
            'name': 'Medium (-250 to 250 HU)',
            'range': (-250, 250),
            'mean_coverage': df['pct_in_medium_window'].mean(),
            'std_coverage': df['pct_in_medium_window'].std(),
            'min_coverage': df['pct_in_medium_window'].min(),
        },
    }
    
    return analysis


def print_window_analysis(analysis):
    """Print window analysis results"""
    print("\n" + "="*80)
    print("WINDOW PARAMETER ANALYSIS")
    print("="*80)
    
    for window_key, window_data in analysis.items():
        print(f"\n{window_data['name']}:")
        print(f"  Range: {window_data['range']}")
        print(f"  Mean tumor coverage: {window_data['mean_coverage']:.1f}% ± {window_data['std_coverage']:.1f}%")
        print(f"  Minimum coverage: {window_data['min_coverage']:.1f}%")
        
        if window_key == 'best_window':
            print(f"  Patients with ≥90% coverage: {window_data['patients_above_90']}")
            print(f"  Patients with ≥95% coverage: {window_data['patients_above_95']}")
    
    print("\n" + "-"*80)
    print("RECOMMENDATION:")
    print("-"*80)
    
    best_coverage = analysis['best_window']['mean_coverage']
    best_min = analysis['best_window']['min_coverage']
    
    if best_coverage >= 95 and best_min >= 85:
        print(f"✓ EXCELLENT: Best window captures {best_coverage:.1f}% of tumor pixels")
        print("  → Use -50 to 300 HU window (same as Pedro's best performing)")
    elif best_coverage >= 90:
        print(f"✓ GOOD: Best window captures {best_coverage:.1f}% of tumor pixels")
        print("  → Use -50 to 300 HU window (acceptable tumor coverage)")
    else:
        print(f"⚠ SUBOPTIMAL: Best window only captures {best_coverage:.1f}% of tumor pixels")
        print("  → Consider adjusting window parameters for CMC dataset")


def generate_visualizations(df, all_hu_samples, output_dir):
    """Generate all visualizations"""
    
    # 1. HU distribution histogram
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    plt.hist(all_hu_samples, bins=100, alpha=0.7, color='navy', edgecolor='black')
    
    # Add window ranges
    windows = [
        {'name': 'Best (-50 to 300)', 'min': -50, 'max': 300, 'color': 'green', 'alpha': 0.2},
        {'name': 'Wide (-500 to 500)', 'min': -500, 'max': 500, 'color': 'blue', 'alpha': 0.1},
        {'name': 'Medium (-250 to 250)', 'min': -250, 'max': 250, 'color': 'red', 'alpha': 0.1},
    ]
    
    for window in windows:
        plt.axvspan(window['min'], window['max'], 
                   alpha=window['alpha'], color=window['color'],
                   label=f"{window['name']} window")
    
    plt.xlabel('Hounsfield Units (HU)', fontsize=12, fontweight='bold')
    plt.ylabel('Frequency (All Patients)', fontsize=12, fontweight='bold')
    plt.title('CMC Dataset: Tumor HU Distribution', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xlim(-200, 500)
    
    # 2. Per-patient mean HU distribution
    plt.subplot(1, 2, 2)
    plt.hist(df['hu_mean'], bins=30, alpha=0.7, color='darkred', edgecolor='black')
    plt.axvline(df['hu_mean'].mean(), color='red', linestyle='--', linewidth=2, 
               label=f'Cohort mean: {df["hu_mean"].mean():.1f} HU')
    plt.xlabel('Mean Tumor HU (per patient)', fontsize=12, fontweight='bold')
    plt.ylabel('Number of Patients', fontsize=12, fontweight='bold')
    plt.title('Inter-Patient HU Variability', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'hu_distribution_histogram.png'), 
                dpi=150, bbox_inches='tight')
    plt.close()
    
    # 2. Window coverage comparison
    plt.figure(figsize=(12, 6))
    
    window_names = ['Best\n(-50 to 300)', 'Medium\n(-250 to 250)', 'Wide\n(-500 to 500)']
    coverage_means = [
        df['pct_in_best_window'].mean(),
        df['pct_in_medium_window'].mean(),
        df['pct_in_wide_window'].mean(),
    ]
    coverage_stds = [
        df['pct_in_best_window'].std(),
        df['pct_in_medium_window'].std(),
        df['pct_in_wide_window'].std(),
    ]
    
    colors = ['lightgreen', 'lightcoral', 'lightblue']
    
    plt.bar(window_names, coverage_means, yerr=coverage_stds, 
           color=colors, edgecolor='black', linewidth=1.5, capsize=5)
    plt.ylabel('Tumor Coverage (%)', fontsize=12, fontweight='bold')
    plt.title('Window Parameter Comparison\n(% of Tumor Pixels Captured)', 
             fontsize=14, fontweight='bold')
    plt.ylim(0, 105)
    plt.axhline(95, color='green', linestyle='--', alpha=0.5, label='95% threshold')
    plt.axhline(90, color='orange', linestyle='--', alpha=0.5, label='90% threshold')
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    
    for i, (mean, std) in enumerate(zip(coverage_means, coverage_stds)):
        plt.text(i, mean + std + 1, f'{mean:.1f}%', 
                ha='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'window_comparison.png'), 
                dpi=150, bbox_inches='tight')
    plt.close()
    
    # 3. Patient variability boxplot
    plt.figure(figsize=(10, 6))
    
    data_to_plot = [df['hu_mean'].values, df['hu_std'].values]
    
    bp = plt.boxplot(data_to_plot, labels=['Mean HU\n(per patient)', 'Std HU\n(within patient)'],
                     patch_artist=True, widths=0.6)
    
    for patch, color in zip(bp['boxes'], ['lightblue', 'lightcoral']):
        patch.set_facecolor(color)
    
    plt.ylabel('Hounsfield Units (HU)', fontsize=12, fontweight='bold')
    plt.title('Inter-Patient vs Intra-Patient HU Variability', 
             fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'patient_variability.png'), 
                dpi=150, bbox_inches='tight')
    plt.close()
    
    print("✓ Generated all visualizations")


def save_summary_report(aggregate_stats, window_analysis, output_dir):
    """Save text summary report"""
    report_path = os.path.join(output_dir, 'aggregate_statistics.txt')
    
    with open(report_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("CMC DATASET HU ANALYSIS - SUMMARY REPORT\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Dataset: {aggregate_stats['n_patients']} patients\n")
        f.write(f"Average tumor volume: {aggregate_stats['mean_tumor_volume']:.0f} voxels\n\n")
        
        f.write("TUMOR HU STATISTICS (Cohort Average):\n")
        f.write("-"*80 + "\n")
        f.write(f"Mean:     {aggregate_stats['cohort_hu_mean']:.2f} HU\n")
        f.write(f"Std:      {aggregate_stats['cohort_hu_std']:.2f} HU\n")
        f.write(f"Median:   {aggregate_stats['cohort_hu_median']:.2f} HU\n")
        f.write(f"Range:    [{aggregate_stats['cohort_hu_min']:.1f}, {aggregate_stats['cohort_hu_max']:.1f}] HU\n\n")
        
        f.write("WINDOW PARAMETER PERFORMANCE:\n")
        f.write("-"*80 + "\n")
        for window_key, window_data in window_analysis.items():
            f.write(f"\n{window_data['name']}:\n")
            f.write(f"  Coverage: {window_data['mean_coverage']:.1f}% ± {window_data['std_coverage']:.1f}%\n")
            f.write(f"  Minimum:  {window_data['min_coverage']:.1f}%\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("RECOMMENDATION:\n")
        f.write("="*80 + "\n")
        best_coverage = window_analysis['best_window']['mean_coverage']
        f.write(f"\nBest window (-50 to 300 HU) captures {best_coverage:.1f}% of tumor pixels.\n")
        f.write("\nConclusion: Use -50 to 300 HU window (consistent with Pedro's methodology)\n")
    
    print(f"✓ Saved summary report: {report_path}")


# ============================================================================
# JUPYTER NOTEBOOK USAGE
# ============================================================================

print("""
================================================================================
CMC DATASET BATCH HU ANALYSIS - JUPYTER VERSION
================================================================================

To run this analysis in Jupyter, use:

    # Set your directory path
    dataset_dir = "/home/radiomicsserver/Downloads/hn_can-main/convert_nifti_format/"
    
    # Run analysis
    df, stats = batch_analyze_dataset(dataset_dir, output_dir="./hu_analysis_results")

This will analyze all 102 patients and generate visualizations.
================================================================================
""")


CMC DATASET BATCH HU ANALYSIS - JUPYTER VERSION

To run this analysis in Jupyter, use:

    # Set your directory path
    dataset_dir = "/home/radiomicsserver/Downloads/hn_can-main/convert_nifti_format/"
    
    # Run analysis
    df, stats = batch_analyze_dataset(dataset_dir, output_dir="./hu_analysis_results")

This will analyze all 102 patients and generate visualizations.

